# Gold - Dimension Players

## Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
sys.path.append("../../..")

import src.gold.dimension.dim_players as p
from src.utils.pyspark_handler import PySparkHandler

## Init spark

In [3]:
if os.getenv('env') != 'dev':
    print('Verify env')
    raise

In [4]:
handler = PySparkHandler(
    app_name="dim_players",
    bucket_endpoint=os.getenv("bucket_endpoint"),
    bucket_access_key=os.getenv("bucket_access_key"),
    bucket_secret_key=os.getenv("bucket_secret_key")
)

2026-10-14 00:24:51,706 INFO src.utils.pyspark_handler - Creating spark session for app 'dim_players'...
2026-10-14 00:25:02,276 INFO src.utils.pyspark_handler - Spark session created


In [5]:
jdbc_url = os.getenv("jdbc_url")
jdbc_user = os.getenv("jdbc_user")
jdbc_password = os.getenv("jdbc_password")

In [6]:
bucket_name = os.getenv("bucket_name")

## Load database

In [7]:
tb_players, tb_atp_player_match, tb_atp_rankings = p.load_tables(handler, bucket_name)

2026-10-14 00:25:02,618 INFO src.gold.dimension.dim_players - Loading dim_players source table...
2026-10-14 00:25:02,620 INFO src.utils.pyspark_handler - Loading data from s3a://dev-tennis-data-lake/silver/tb_atp_players/ (format=parquet)...
2026-10-14 00:25:10,164 INFO src.utils.pyspark_handler - Data loaded from s3a://dev-tennis-data-lake/silver/tb_atp_players/
2026-10-14 00:25:10,165 INFO src.utils.pyspark_handler - Loading data from s3a://dev-tennis-data-lake/silver/tb_atp_player_match/ (format=parquet)...
2026-10-14 00:25:11,790 INFO src.utils.pyspark_handler - Data loaded from s3a://dev-tennis-data-lake/silver/tb_atp_player_match/
2026-10-14 00:25:11,790 INFO src.utils.pyspark_handler - Loading data from s3a://dev-tennis-data-lake/silver/tb_atp_rankings/ (format=parquet)...
2026-10-14 00:25:13,427 INFO src.utils.pyspark_handler - Data loaded from s3a://dev-tennis-data-lake/silver/tb_atp_rankings/


## Run transformation

In [29]:
df_final = p.run_transformation(tb_players, tb_atp_player_match, tb_atp_rankings)

2026-10-14 01:29:48,834 INFO src.gold.dimension.dim_players - dim_players transformation completed


In [34]:
tb_atp_rankings.limit(2)

2026-10-14 01:34:44,347 INFO py4j.clientserver - Closing down clientserver connection


DATE_WEEK_RANKING,COD_PLAYER_ID,DES_PLAYER_NAME,NUM_PLAYER_RANK,NUM_PLAYER_AGE,NUM_PLAYER_RANK_PTS,NUM_PLAYER_LE_PTS,NUM_PLAYER_TOURNEY_PLAYED,NUM_PLAYER_DROP_PTS,NUM_PLAYER_NEXT_BEST,DATE_INGESTION
1973-08-20,G075,Andres Gimeno,24,NULL,36,NULL,NULL,NULL,NULL,2026-10-10
1973-08-20,A063,Arthur Ashe,4,NULL,221,NULL,NULL,NULL,NULL,2026-10-10


In [42]:
tb_atp_rankings.select("COD_PLAYER_ID", "DES_PLAYER_NAME").where("DES_PLAYER_NAME = 'Robert Phillips'").distinct()

COD_PLAYER_ID,DES_PLAYER_NAME
PD13,Robert Phillips
P239,Robert Phillips


In [37]:
import pyspark.sql.functions as f

tb_atp_rankings.select("COD_PLAYER_ID", "DES_PLAYER_NAME").groupby("DES_PLAYER_NAME").agg(f.count_distinct("*").alias("a")).where("a > 1")

DES_PLAYER_NAME,a
Iker Ortiz,2
Jose Wasserfirer,2
Robert Phillips,2
Alberto Gonzalez,2
Luis Perez,2
Fred Viancos,2
Paul Mick,2
Claudio Semmelmann,2
Jose Demeterco,2
Peter Dellavedova,2


## Save dataframe

In [15]:
# p.save_table(handler, df_final, bucket_name, jdbc_url, jdbc_user, jdbc_password)